In [1]:
import json

# ==========================================
# 0. FONCTION DE SÉCURITÉ (PRIORITÉ AU FRANÇAIS)
# ==========================================
def extraire_texte(valeur):
    """
    Extrait le texte en ciblant spécifiquement la langue française ("fr")
    si la donnée est un dictionnaire multilingue.
    """
    if isinstance(valeur, dict) and valeur:
        if 'fr' in valeur:
            return str(valeur['fr'])
        else:
            return str(list(valeur.values())[0])
    elif isinstance(valeur, list) and valeur:
        return str(valeur[0])
    elif isinstance(valeur, str):
        return valeur
    return None

# ==========================================
# 1. CONFIGURATION DES FICHIERS
# ==========================================
fichier_principal = 'data.json'
fichier_secondaire = 'theses-soutenues.json' # Mis à jour avec ton nom de fichier
fichier_sortie = 'data_enrichie.json'

# ==========================================
# 2. CHARGEMENT DES DONNÉES
# ==========================================
print("Chargement des fichiers...")
with open(fichier_principal, 'r', encoding='utf-8') as f1:
    data_principale = json.load(f1)

with open(fichier_secondaire, 'r', encoding='utf-8') as f2:
    data_secondaire = json.load(f2)

# ==========================================
# 3. CRÉATION DE L'INDEX DE RECHERCHE
# ==========================================
index_recherche = {}

for item in data_secondaire:
    titre_brut = item.get('titres') 
    titre_texte = extraire_texte(titre_brut)
    
    if titre_texte:
        cle = titre_texte[:40].lower()
        # En cas de doublons sur les 40 premiers caractères dans theses-soutenues.json,
        # on pourrait écraser la précédente. On garde la logique simple ici.
        index_recherche[cle] = item

# ==========================================
# 4. JOINTURE AVEC VÉRIFICATION DE L'ANNÉE
# ==========================================
theses_mises_a_jour = 0
theses_ignorees_annee = 0

for item in data_principale:
    # 1. Initialisation par défaut
    item['date_soutenance'] = None
    item['resume'] = None
    
    # 2. Extraction du titre et de l'année de référence (data.json)
    titre_brut = item.get('titre')
    titre_texte = extraire_texte(titre_brut)
    
    annee_data_str = str(item.get('annee', '0'))
    annee_data = int(annee_data_str) if annee_data_str.isdigit() else 0
    
    if titre_texte:
        cle = titre_texte[:40].lower()
        
        # 3. Si on trouve une correspondance dans l'index
        if cle in index_recherche:
            correspondance = index_recherche[cle]
            date_trouvee = correspondance.get('date_soutenance')
            
            # --- Vérification de l'année ---
            annee_soutenance = 0
            # Si une date existe et a au moins 4 caractères (ex: "1996-01-01")
            if date_trouvee and len(str(date_trouvee)) >= 4:
                try:
                    # On extrait les 4 premiers caractères (YYYY)
                    annee_soutenance = int(str(date_trouvee)[:4])
                except ValueError:
                    pass
            
            # Si on a bien récupéré les deux années, et que la soutenance est AVANT l'année data.json
            if annee_soutenance > 0 and annee_data > 0 and annee_soutenance < annee_data:
                theses_ignorees_annee += 1
                continue # On ignore cette correspondance et on passe à la thèse suivante !
            
            # --- Si l'année est valide (ou manquante), on procède à la mise à jour ---
            resume_brut = correspondance.get('resumes')
            resume_trouve = extraire_texte(resume_brut)
            
            if date_trouvee:
                item['date_soutenance'] = date_trouvee
            if resume_trouve:
                item['resume'] = resume_trouve
                
            if date_trouvee or resume_trouve:
                theses_mises_a_jour += 1

# ==========================================
# 5. SAUVEGARDE DU NOUVEAU FICHIER JSON
# ==========================================
with open(fichier_sortie, 'w', encoding='utf-8') as f_out:
    json.dump(data_principale, f_out, ensure_ascii=False, indent=2)

print(f"Jointure terminée avec filtrage temporel (basée sur 40 caractères) !")
print(f"- {theses_mises_a_jour} thèses enrichies avec succès.")
print(f"- {theses_ignorees_annee} correspondances ont été ignorées car la date de soutenance était trop ancienne.")
print(f"- Fichier sauvegardé : {fichier_sortie}")

Chargement des fichiers...
Jointure terminée avec filtrage temporel (basée sur 40 caractères) !
- 3174 thèses enrichies avec succès.
- 50 correspondances ont été ignorées car la date de soutenance était trop ancienne.
- Fichier sauvegardé : data_enrichie.json
